# U-Net from scratch — Binary segmentation on COCO
**Introduction to Deep Learning — Project**

The goal of this project is to implement a **U-Net** convolutional neural network
**from scratch** with **PyTorch** (no Lightning, no pre-trained model and no
existing U-Net implementation) and to use it to **segment objects from the COCO
dataset**. We only separate the **foreground** (any object) from the
**background**: this is *binary* segmentation, no semantic class is predicted
(semantic segmentation is mentioned as a possible bonus).

What we respect from the subject:
- the input is an RGB image (3 channels) and the output mask has the same size
  as the input (we use padded convolutions);
- the model is evaluated on a validation split (unseen data) at every epoch;
- only PyTorch is used (no Lightning), and the U-Net is implemented from scratch
  (no pre-trained model, no existing implementation);
- the choices are explained in the markdown cells below.

### How to run
- **With COCO** (training PC / Kaggle / Colab): in the *Configuration* cell keep
  `use_coco = True` and set the paths to the COCO 2017 images and annotation
  files, then **Run all**.
- **Without COCO** (quick test): set `use_coco = False` (or just leave the paths
  empty). A small synthetic *shapes* dataset is generated on the fly so the whole
  notebook still runs end to end and produces curves and segmentations.

In [ ]:
import os, json, copy, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageDraw
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
%matplotlib inline

## 1. Dataset

We build a **binary** mask from the COCO instance annotations: every annotated
object becomes **foreground (1)**, the rest is **background (0)**. No class label
is kept, which is exactly what the subject asks for.

`CocoBinaryDataset` returns, for each sample:
- the **image**: a normalized `(3, H, W)` tensor (ImageNet mean/std),
- the **mask**: a `(1, H, W)` tensor with values in `{0, 1}` and the **same size**
  as the image.

`SyntheticShapes` generates random circles/rectangles with their ground-truth
mask. It needs no download and is used to test the pipeline quickly and as a
fallback when COCO is not available.

In [ ]:
import os

import numpy as np
import torch
from PIL import Image, ImageDraw
from torch.utils.data import Dataset

MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]


def _to_input_tensor(pil_img, img_size):
    pil_img = pil_img.resize((img_size, img_size), Image.BILINEAR)
    arr = np.asarray(pil_img, dtype=np.float32) / 255.0
    arr = (arr - np.array(MEAN, dtype=np.float32)) / np.array(STD, dtype=np.float32)
    return torch.from_numpy(arr.transpose(2, 0, 1)).contiguous()


def _to_mask_tensor(mask_arr, img_size):
    mask = Image.fromarray((mask_arr * 255).astype(np.uint8))
    mask = mask.resize((img_size, img_size), Image.NEAREST)
    mask = (np.asarray(mask, dtype=np.float32) > 127).astype(np.float32)
    return torch.from_numpy(mask)[None]


class CocoBinaryDataset(Dataset):
    def __init__(self, img_dir, ann_file, img_size=128, max_samples=None):
        from pycocotools.coco import COCO

        self.coco = COCO(ann_file)
        self.img_dir = img_dir
        self.img_size = img_size

        ids = sorted(self.coco.getImgIds())
        ids = [i for i in ids if len(self.coco.getAnnIds(imgIds=i, iscrowd=None)) > 0]
        if max_samples is not None:
            ids = ids[:max_samples]
        self.ids = ids

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        info = self.coco.loadImgs(img_id)[0]
        path = os.path.join(self.img_dir, info["file_name"])
        image = Image.open(path).convert("RGB")

        ann_ids = self.coco.getAnnIds(imgIds=img_id, iscrowd=None)
        anns = self.coco.loadAnns(ann_ids)
        mask = np.zeros((info["height"], info["width"]), dtype=np.uint8)
        for ann in anns:
            mask = np.maximum(mask, self.coco.annToMask(ann))

        return _to_input_tensor(image, self.img_size), _to_mask_tensor(mask, self.img_size)


class SyntheticShapes(Dataset):
    def __init__(self, n_samples=256, img_size=128, seed=0):
        self.n_samples = n_samples
        self.img_size = img_size
        self.seed = seed

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        rng = np.random.default_rng(self.seed * 100000 + idx)
        s = self.img_size

        bg = rng.integers(0, 120, size=3)
        img = np.ones((s, s, 3), dtype=np.float32) * bg
        img += rng.normal(0, 10, size=(s, s, 3)).astype(np.float32)

        pil = Image.fromarray(np.clip(img, 0, 255).astype(np.uint8))
        draw = ImageDraw.Draw(pil)
        mask_pil = Image.fromarray(np.zeros((s, s), dtype=np.uint8))
        mdraw = ImageDraw.Draw(mask_pil)

        for _ in range(rng.integers(1, 4)):
            color = tuple(int(c) for c in rng.integers(140, 256, size=3))
            x0, y0 = rng.integers(0, s - s // 4, size=2)
            w, h = rng.integers(s // 6, s // 2, size=2)
            x1, y1 = min(x0 + w, s - 1), min(y0 + h, s - 1)
            if rng.random() < 0.5:
                draw.ellipse([x0, y0, x1, y1], fill=color)
                mdraw.ellipse([x0, y0, x1, y1], fill=1)
            else:
                draw.rectangle([x0, y0, x1, y1], fill=color)
                mdraw.rectangle([x0, y0, x1, y1], fill=1)

        mask = np.asarray(mask_pil, dtype=np.float32)
        return _to_input_tensor(pil, s), _to_mask_tensor(mask, s)

## 2. Utilities

Reproducibility (seeding), device selection (CUDA on the training PC, MPS on mac,
else CPU) and a few visualization helpers.

In [ ]:
import random

import numpy as np
import torch



def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def denormalize(img_tensor):
    img = img_tensor.detach().cpu().numpy().transpose(1, 2, 0)
    img = img * np.array(STD) + np.array(MEAN)
    return np.clip(img, 0, 1)


def plot_history(history, ax=None):
    import matplotlib.pyplot as plt

    if ax is None:
        _, ax = plt.subplots(1, 2, figsize=(11, 4))
    epochs = range(1, len(history["train_loss"]) + 1)
    ax[0].plot(epochs, history["train_loss"], label="train")
    ax[0].plot(epochs, history["val_loss"], label="val")
    ax[0].set_title("Loss")
    ax[0].set_xlabel("epoch")
    ax[0].legend()
    ax[1].plot(epochs, history["val_dice"], label="val Dice", color="green")
    ax[1].plot(epochs, history["val_iou"], label="val IoU", color="orange")
    ax[1].set_title("Validation metrics")
    ax[1].set_xlabel("epoch")
    ax[1].legend()
    return ax


@torch.no_grad()
def show_predictions(model, dataset, device, n=4, threshold=0.5, seed=0):
    import matplotlib.pyplot as plt

    model.eval()
    rng = np.random.default_rng(seed)
    indices = rng.choice(len(dataset), size=min(n, len(dataset)), replace=False)

    fig, axes = plt.subplots(len(indices), 3, figsize=(9, 3 * len(indices)))
    if len(indices) == 1:
        axes = axes[None, :]
    for row, idx in enumerate(indices):
        image, mask = dataset[int(idx)]
        logits = model(image[None].to(device))
        pred = (torch.sigmoid(logits)[0, 0].cpu().numpy() > threshold).astype(float)

        axes[row, 0].imshow(denormalize(image))
        axes[row, 0].set_title("image")
        axes[row, 1].imshow(mask[0], cmap="gray")
        axes[row, 1].set_title("ground truth")
        axes[row, 2].imshow(pred, cmap="gray")
        axes[row, 2].set_title("prediction")
        for c in range(3):
            axes[row, c].axis("off")
    fig.tight_layout()
    return fig

## 3. Configuration

Set `use_coco = True` and fill the COCO paths to train on COCO. Otherwise the
synthetic dataset is used. The `SMOKE` environment variable shrinks everything to
test the notebook in a few seconds.

In [ ]:
config = {
    "use_coco": True,
    "coco_train_images": "data/coco/train2017",
    "coco_train_ann":    "data/coco/annotations/instances_train2017.json",
    "coco_val_images":   "data/coco/val2017",
    "coco_val_ann":      "data/coco/annotations/instances_val2017.json",
    "img_size": 128,
    "batch_size": 16,
    "num_workers": 2,
    "epochs": 30,
    "base": 64,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "patience": 6,
    "max_train": None,
    "max_val": None,
}

if os.environ.get("SMOKE") == "1":
    config.update(use_coco=False, img_size=64, batch_size=8, epochs=5,
                  base=8, max_train=96, max_val=32, num_workers=0)

set_seed(42)
device = get_device()
print("device:", device)
print("config:", config)

In [ ]:
paths = [config["coco_train_images"], config["coco_train_ann"],
         config["coco_val_images"], config["coco_val_ann"]]
coco_ready = config["use_coco"] and all(p and os.path.exists(p) for p in paths)

if coco_ready:
    print("Loading COCO ...")
    train_ds = CocoBinaryDataset(config["coco_train_images"], config["coco_train_ann"],
                                 img_size=config["img_size"], max_samples=config["max_train"])
    val_ds = CocoBinaryDataset(config["coco_val_images"], config["coco_val_ann"],
                               img_size=config["img_size"], max_samples=config["max_val"])
else:
    if config["use_coco"]:
        print("COCO paths not found -> falling back to the synthetic dataset.")
    else:
        print("Using the synthetic dataset.")
    train_ds = SyntheticShapes(n_samples=config["max_train"] or 256,
                               img_size=config["img_size"], seed=0)
    val_ds = SyntheticShapes(n_samples=config["max_val"] or 64,
                             img_size=config["img_size"], seed=1)

train_loader = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True,
                          num_workers=config["num_workers"])
val_loader = DataLoader(val_ds, batch_size=config["batch_size"], shuffle=False,
                        num_workers=config["num_workers"])
print(f"train: {len(train_ds)} images | val (unseen): {len(val_ds)} images")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(11, 6))
for i in range(4):
    img, msk = train_ds[i]
    axes[0, i].imshow(denormalize(img)); axes[0, i].set_title("image"); axes[0, i].axis("off")
    axes[1, i].imshow(msk[0], cmap="gray"); axes[1, i].set_title("mask"); axes[1, i].axis("off")
plt.tight_layout(); plt.show()

## 4. The U-Net model (from scratch)

U-Net is an **encoder–decoder** network with **skip connections**:
- the **encoder** (contracting path) extracts features while reducing the spatial
  resolution with max-pooling;
- the **decoder** (expanding path) increases the resolution back with **transposed
  convolutions** (the "up-convolution" seen in the course) until the output has the
  same size as the input;
- **skip connections** copy the encoder feature maps to the decoder so the fine
  spatial details lost during pooling are recovered.

**Implementation choices**
- each block is a `DoubleConv` = (Conv 3×3 → BatchNorm → ReLU) ×2. BatchNorm is put
  **before** the activation, as seen for residual networks in the course, and the
  conv bias is disabled because BatchNorm already adds one.
- we use **padded ("same") convolutions** so the output mask has exactly the same
  height/width as the input image.
- the last layer is a 1×1 convolution producing a **single channel** (foreground
  logit). The sigmoid is **not** applied here; it is included in the loss
  (`BCEWithLogitsLoss`) for numerical stability.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=1, base=64):
        super().__init__()
        self.pool = nn.MaxPool2d(2)

        self.enc1 = DoubleConv(in_channels, base)
        self.enc2 = DoubleConv(base, base * 2)
        self.enc3 = DoubleConv(base * 2, base * 4)
        self.enc4 = DoubleConv(base * 4, base * 8)

        self.bottleneck = DoubleConv(base * 8, base * 16)

        self.up4 = nn.ConvTranspose2d(base * 16, base * 8, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(base * 16, base * 8)
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(base * 2, base)

        self.head = nn.Conv2d(base, num_classes, kernel_size=1)

    @staticmethod
    def _concat(upsampled, skip):
        diff_y = skip.size(2) - upsampled.size(2)
        diff_x = skip.size(3) - upsampled.size(3)
        upsampled = F.pad(
            upsampled,
            [diff_x // 2, diff_x - diff_x // 2, diff_y // 2, diff_y - diff_y // 2],
        )
        return torch.cat([skip, upsampled], dim=1)

    def forward(self, x):
        c1 = self.enc1(x)
        c2 = self.enc2(self.pool(c1))
        c3 = self.enc3(self.pool(c2))
        c4 = self.enc4(self.pool(c3))

        b = self.bottleneck(self.pool(c4))

        d4 = self.dec4(self._concat(self.up4(b), c4))
        d3 = self.dec3(self._concat(self.up3(d4), c3))
        d2 = self.dec2(self._concat(self.up2(d3), c2))
        d1 = self.dec1(self._concat(self.up1(d2), c1))

        return self.head(d1)

In [ ]:
_net = UNet(base=config["base"])
_x = torch.randn(1, 3, config["img_size"], config["img_size"])
print("input :", tuple(_x.shape))
print("output:", tuple(_net(_x).shape))
print("params:", sum(p.numel() for p in _net.parameters()))
del _net, _x

## 5. Loss and metrics

- **Loss:** a combination of **Binary Cross-Entropy** (per-pixel classification)
  and **Dice loss** (overlap of the masks). BCE gives a stable gradient everywhere,
  Dice handles the foreground/background imbalance — together they train better
  than either one alone.
- **Metrics** (on the validation set): **Dice** coefficient, **IoU** (Jaccard) and
  **pixel accuracy**. Dice and IoU are the standard segmentation metrics; accuracy
  alone is misleading when the background dominates the image.

In [ ]:
import torch
import torch.nn as nn


def dice_loss(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    probs = probs.view(probs.size(0), -1)
    targets = targets.view(targets.size(0), -1)
    intersection = (probs * targets).sum(dim=1)
    union = probs.sum(dim=1) + targets.sum(dim=1)
    dice = (2 * intersection + eps) / (union + eps)
    return 1 - dice.mean()


class BCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.5):
        super().__init__()
        self.bce_weight = bce_weight
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        bce = self.bce(logits, targets)
        dice = dice_loss(logits, targets)
        return self.bce_weight * bce + (1 - self.bce_weight) * dice


@torch.no_grad()
def dice_coeff(logits, targets, threshold=0.5, eps=1e-6):
    preds = (torch.sigmoid(logits) > threshold).float()
    preds = preds.view(preds.size(0), -1)
    targets = targets.view(targets.size(0), -1)
    inter = (preds * targets).sum(dim=1)
    union = preds.sum(dim=1) + targets.sum(dim=1)
    return ((2 * inter + eps) / (union + eps)).mean().item()


@torch.no_grad()
def iou_score(logits, targets, threshold=0.5, eps=1e-6):
    preds = (torch.sigmoid(logits) > threshold).float()
    preds = preds.view(preds.size(0), -1)
    targets = targets.view(targets.size(0), -1)
    inter = (preds * targets).sum(dim=1)
    union = preds.sum(dim=1) + targets.sum(dim=1) - inter
    return ((inter + eps) / (union + eps)).mean().item()


@torch.no_grad()
def pixel_accuracy(logits, targets, threshold=0.5):
    preds = (torch.sigmoid(logits) > threshold).float()
    return (preds == targets).float().mean().item()

## 6. Training

We follow the loop structure seen in the course:

```
for _ in range(epochs):
    train_loop(train_dataset)   # forward + backward
    test_loop(test_dataset)     # forward only, on unseen data
    process_statistics()        # store losses / metrics
```

**Choices and regularization techniques (from the course)**
- **Optimizer:** Adam, a robust default.
- **Weight decay (L2 regularization):** a small penalty on the weights to reduce
  overfitting.
- **Early stopping:** monitor the validation loss and stop after `patience` epochs
  without improvement, keeping the best weights.
- **LR scheduling:** `ReduceLROnPlateau` lowers the learning rate when the
  validation loss plateaus.
- the model is **evaluated on the validation set (unseen)** at every epoch to
  measure generalization, as required.

In [ ]:
import copy

import torch
from tqdm.auto import tqdm



def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    running = 0.0
    for images, masks in tqdm(loader, leave=False, desc="train"):
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = loss_fn(logits, masks)
        loss.backward()
        optimizer.step()
        running += loss.item() * images.size(0)
    return running / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, loss_fn, device):
    model.eval()
    loss_sum = dice_sum = iou_sum = acc_sum = 0.0
    for images, masks in tqdm(loader, leave=False, desc="val"):
        images, masks = images.to(device), masks.to(device)
        logits = model(images)
        bs = images.size(0)
        loss_sum += loss_fn(logits, masks).item() * bs
        dice_sum += dice_coeff(logits, masks) * bs
        iou_sum += iou_score(logits, masks) * bs
        acc_sum += pixel_accuracy(logits, masks) * bs
    n = len(loader.dataset)
    return {"loss": loss_sum / n, "dice": dice_sum / n, "iou": iou_sum / n, "acc": acc_sum / n}


def fit(model, train_loader, val_loader, loss_fn, optimizer, device,
        epochs=20, patience=5, scheduler=None, verbose=True):
    history = {"train_loss": [], "val_loss": [], "val_dice": [], "val_iou": [], "val_acc": []}
    best_val = float("inf")
    best_weights = copy.deepcopy(model.state_dict())
    epochs_without_improve = 0

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
        val = evaluate(model, val_loader, loss_fn, device)
        if scheduler is not None:
            scheduler.step(val["loss"])

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val["loss"])
        history["val_dice"].append(val["dice"])
        history["val_iou"].append(val["iou"])
        history["val_acc"].append(val["acc"])

        if verbose:
            print(
                f"epoch {epoch:3d}/{epochs} | train loss {train_loss:.4f} | "
                f"val loss {val['loss']:.4f} | dice {val['dice']:.4f} | "
                f"iou {val['iou']:.4f} | acc {val['acc']:.4f}"
            )

        if val["loss"] < best_val - 1e-4:
            best_val = val["loss"]
            best_weights = copy.deepcopy(model.state_dict())
            epochs_without_improve = 0
        else:
            epochs_without_improve += 1
            if epochs_without_improve >= patience:
                if verbose:
                    print(f"early stopping at epoch {epoch} (best val loss {best_val:.4f})")
                break

    model.load_state_dict(best_weights)
    return history

In [ ]:
model = UNet(in_channels=3, num_classes=1, base=config["base"]).to(device)
loss_fn = BCEDiceLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"],
                             weight_decay=config["weight_decay"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3)

history = fit(model, train_loader, val_loader, loss_fn, optimizer, device,
              epochs=config["epochs"], patience=config["patience"], scheduler=scheduler)

## 7. Results

Training / validation curves, final metrics on the unseen validation set, and a few predicted segmentations.

In [ ]:
plot_history(history)
plt.show()

In [ ]:
final = evaluate(model, val_loader, loss_fn, device)
print("Final validation metrics:", {k: round(v, 4) for k, v in final.items()})

In [ ]:
fig = show_predictions(model, val_ds, device, n=4)
plt.show()

## 8. Conclusion

We implemented a **U-Net from scratch** in PyTorch and trained it for **binary
foreground/background segmentation** on COCO. We reused several notions from the
course: **transposed convolutions** for the upsampling path, **skip connections**,
**batch normalization** before the activation, and **regularization** (weight
decay, early stopping, LR scheduling). The model is always evaluated on **unseen**
data, and the predicted mask has the **same size** as the input image.

**Possible improvements / bonus**
- **data augmentation** (random flips, crops, color jitter) to regularize further;
- replace the `DoubleConv` blocks by **residual blocks** (also seen in the course)
  for a deeper, more stable network;
- add **dropout** in the bottleneck;
- extend to **semantic** segmentation (one output channel per class) — the bonus
  mentioned in the subject.